# Домашнее задание 5

In [1]:
import heapq
from collections import deque

class TreeNode:
    """Класс узла бинарного дерева."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
        # Добавляем поле для 5-й задачи
        self.balance_factor = 0

def build_tree(arr, i=0):
    """Вспомогательная функция для построения дерева из массива."""
    if i >= len(arr) or arr[i] is None:
        return None
    root = TreeNode(arr[i])
    root.left = build_tree(arr, 2 * i + 1)
    root.right = build_tree(arr, 2 * i + 2)
    return root

def print_tree(root):
    """Вспомогательная функция для вывода дерева по уровням."""
    if not root: return "[]"
    queue = deque([root])
    result = []
    while queue:
        node = queue.popleft()
        if node:
            result.append(str(node.val))
            queue.append(node.left)
            queue.append(node.right)
        else:
            result.append("null")
    while result and result[-1] == "null":
        result.pop()
    print(result)

# Задача 1. Полное бинарное дерево
Реализуйте функцию, которая проверяет, является ли заданное бинарное дерево
полным.
Необходимо корректно определить условие заполненности уровней и объяснить
выбранный способ обхода (например, BFS).

Объяснение: Полное бинарное дерево — это дерево, в котором все уровни, кроме, возможно, последнего, полностью заполнены, а все узлы последнего уровня находятся как можно левее. Используем BFS (обход в ширину). Мы идем по уровням. Как только мы встретили None (пустой узел), все последующие узлы в очереди тоже должны быть None. Если мы встречаем None, а потом снова число — дерево не полное.

In [3]:
def is_complete_tree(root: TreeNode) -> bool:
    if not root:
        return True
    
    queue = deque([root])
    seen_none = False # Флаг: встретили ли мы пустой узел
    
    while queue:
        node = queue.popleft()
        
        if node is None:
            seen_none = True
        else:
            # Если мы уже видели None, но сейчас встретили число - дерево не полное
            if seen_none:
                return False
            # Добавляем детей в очередь (даже если они None)
            queue.append(node.left)
            queue.append(node.right)
            
    return True

# Тесты
print("Проверка на полное дерево:")
tree_complete = build_tree([1, 2, 3, 4, 5, 6])
tree_not_complete = build_tree([1, 2, 3, 4, None, 6])
print(f"Дерево [1, 2, 3, 4, 5, 6] полное? {is_complete_tree(tree_complete)}") # True
print(f"Дерево [1, 2, 3, 4, None, 6] полное? {is_complete_tree(tree_not_complete)}") # False

Проверка на полное дерево:
Дерево [1, 2, 3, 4, 5, 6] полное? True
Дерево [1, 2, 3, 4, None, 6] полное? False


# Задача 2. Объединение K отсортированных массивов
Напишите функцию, объединяющую K отсортированных массивов в один
отсортированный массив.
Для эффективного решения используйте мин-кучу и проанализируйте сложность
алгоритма.


Объяснение: У нас есть K массивов. Мы хотим объединить их в один отсортированный.Используем мин-кучу (Min-Heap).
Мы кладем в кучу первый элемент из каждого массива (вместе с его индексом в массиве). Куча всегда выдает нам самый маленький элемент. Мы берем его, добавляем в результат, а затем сразу же добавляем в кучу следующий элемент из того же массива, откуда мы взяли текущий. Сложность: если всего N элементов, каждое добавление/извлечение из кучи занимает O(logK). Итого O(NlogK).

In [5]:
def merge_k_sorted_arrays(arrays: list[list[int]]) -> list[int]:
    result = []
    min_heap = []
    
    # Кладем в кучу первый элемент из каждого массива
    # Формат в куче: (значение, индекс_массива, индекс_элемента_в_массиве)
    for i, arr in enumerate(arrays):
        if arr:
            heapq.heappush(min_heap, (arr[0], i, 0))
            
    while min_heap:
        val, arr_idx, elem_idx = heapq.heappop(min_heap)
        result.append(val)
        
        # Если в этом массиве есть еще элементы, добавляем следующий
        if elem_idx + 1 < len(arrays[arr_idx]):
            next_val = arrays[arr_idx][elem_idx + 1]
            heapq.heappush(min_heap, (next_val, arr_idx, elem_idx + 1))
            
    return result

# Тест
print("Объединение K массивов:")
arrays = [[1, 4, 5], [1, 3, 4], [2, 6]]
print(f"Массивы: {arrays}")
print(f"Результат: {merge_k_sorted_arrays(arrays)}") # [1, 1, 2, 3, 4, 4, 5, 6]

Объединение K массивов:
Массивы: [[1, 4, 5], [1, 3, 4], [2, 6]]
Результат: [1, 1, 2, 3, 4, 4, 5, 6]


# Задача 3. K-ый наименьший / наибольший элемент
Дан массив чисел. Необходимо найти K-ый наименьший или K-ый наибольший
элемент.
Рассмотрите эффективные подходы (например, с использованием кучи) и обоснуйте
сложность решения.


Объяснение: Найти K-й наименьший элемент в обычном массиве можно разными способами (например, отсортировать и взять по индексу за 
O(nlogn)). Но эффективнее использовать мин-кучу размера K. Мы проходим по массиву и кладем элементы в кучу. Если размер кучи превышает K, мы выкидываем самый большой элемент из кучи. В итоге в куче останутся ровно K наименьших элементов, а самый маленький из них (вершина кучи) и будет ответом. Сложность O(nlogK).

In [6]:
def find_kth_smallest(nums: list[int], k: int) -> int:
    """Находит K-й наименьший элемент с помощью кучи."""
    if not nums or k <= 0 or k > len(nums):
        return -1
        
    # В Python heapq реализует мин-кучу. Чтобы имитировать макс-кучу (для хранения K наименьших), мы будем хранить отрицательные числа.
    max_heap = []
    
    for num in nums:
        heapq.heappush(max_heap, -num)
        # Если куча стала больше K, выкидываем самый большой (самый "отрицательный")
        if len(max_heap) > k:
            heapq.heappop(max_heap)
            
    # В конце вершина кучи (с минусом) - это K-й наименьший
    return -max_heap[0]

# Тест
print("К-й наименьший элемент в массиве:")
nums = [3, 2, 1, 5, 6, 4]
k = 2
print(f"Массив: {nums}, K = {k}")
print(f"{k}-й наименьший элемент: {find_kth_smallest(nums, k)}") # Ожидается 2

К-й наименьший элемент в массиве:
Массив: [3, 2, 1, 5, 6, 4], K = 2
2-й наименьший элемент: 2


# Задача 4. K-ый наименьший / наибольший элемент в BST
Дано бинарное дерево поиска.
Реализуйте алгоритм нахождения K-ого наименьшего или наибольшего элемента,
используя свойства BST и обход дерева.

Объяснение: Главное свойство бинарного дерева поиска (BST) — если сделать симметричный обход (Left -> Node -> Right), то значения выведутся в отсортированном по возрастанию порядке. Значит, чтобы найти K-й наименьший, нам просто нужно сделать обход в глубину (DFS) в этом порядке и остановиться, когда счетчик дойдет до K.

In [7]:
def kth_smallest_bst(root: TreeNode, k: int) -> int:
    """Находит K-й наименьший элемент в BST (симметричный обход)."""
    stack = []
    current = root
    
    while stack or current:
        # Идем максимально влево
        while current:
            stack.append(current)
            current = current.left
            
        # Берем узел из стека
        current = stack.pop()
        k -= 1
        
        # Если это K-й элемент, возвращаем его
        if k == 0:
            return current.val
            
        # Переходим к правому поддереву
        current = current.right
        
    return -1

# Тест
print("К-й наименьший элемент в BST:")
bst_root = build_tree([5, 3, 6, 2, 4, None, None, 1])
k = 3
print(f"BST: {[5, 3, 6, 2, 4, None, None, 1]}, K = {k}")
print(f"{k}-й наименьший элемент: {kth_smallest_bst(bst_root, k)}") # Ожидается 3

К-й наименьший элемент в BST:
BST: [5, 3, 6, 2, 4, None, None, 1], K = 3
3-й наименьший элемент: 3


# Задача 5. Balance factor
На вход функции передаётся указатель на корень бинарного дерева поиска.
Необходимо для каждого узла вычислить и заполнить поле balance_factor (разницу
высот левого и правого поддеревьев), объяснив способ вычисления высоты.


Объяснение: Фактор баланса узла — это разница между высотой его левого и правого поддеревьев (height(left) - height(right)). Нам нужно обойти всё дерево (снизу вверх, через DFS), вычислить высоту каждого узла и записать в его поле balance_factor. Мы используем пост-ордерный обход (снизу вверх). Функция сначала рекурсивно вычисляет высоту левого и правого поддеревьев. Высота пустого узла равна 0, а листа — 1. Для текущего узла мы записываем разницу высот в balance_factor и возвращаем наверх его собственную высоту, которая равна 1 + max(left_height, right_height). Это позволяет за один проход O(n) вычислить высоты и заполнить все факторы баланса

In [9]:
def calculate_balance_factors(root: TreeNode) -> int:
    """Рекурсивно вычисляет высоту и заполняет balance_factor для всех узлов."""
    if not root:
        return 0
        
    # Сначала вычисляем высоты левого и правого поддеревьев
    left_height = calculate_balance_factors(root.left)
    right_height = calculate_balance_factors(root.right)
    
    # Заполняем поле в узле
    root.balance_factor = left_height - right_height
    
    # Возвращаем высоту текущего узла (1 + максимальная высота из детей)
    return 1 + max(left_height, right_height)

def print_balance_factors(root):
    if not root: return
    print_balance_factors(root.left)
    print(f"Узел {root.val}: balance_factor = {root.balance_factor}")
    print_balance_factors(root.right)

# Тест
print("Вычисление Balance Factor:")
tree_bf = build_tree([1, 2, 3, 4])
calculate_balance_factors(tree_bf)
print_balance_factors(tree_bf)
# Узел 4: 0 (лист)
# Узел 2: 1 (слева высота 1, справа 0 -> 1 - 0 = 1)
# Узел 3: 0 (лист)
# Узел 1: 1 (слева высота 2, справа 1 -> 2 - 1 = 1)

Вычисление Balance Factor:
Узел 4: balance_factor = 0
Узел 2: balance_factor = 1
Узел 1: balance_factor = 1
Узел 3: balance_factor = 0


# Задача 6. Преобразование в зеркальное дерево
Реализуйте алгоритм зеркального отражения бинарного дерева, меняя местами
левые и правые поддеревья каждого узла.
Поясните, почему алгоритм корректен и какова его временная сложность.

Объяснение: Нам нужно поменять местами левых и правых потомков у каждого узла дерева. Принцип работы -  заходим в узел, меняем его left и right местами, а затем рекурсивно вызываем эту же функцию для нового левого и нового правого ребенка.Временная сложность алгоритма зеркального отражения дерева — O(n), где n — это количество узлов в дереве. Потому что: 1. Посещение каждого узла: Функция вызывается ровно один раз для каждого существующего узла в дереве. Она не пропускает ни одного узла, потому что должна поменять местами left и right у каждого из них. 2. Действия внутри узла: Внутри каждого вызова мы делаем операцию обмена ссылками (root.left, root.right = root.right, root.left). Это действие выполняется за константное время O(1). Итого O(n).

In [10]:
def invert_tree(root: TreeNode) -> TreeNode:
    """Зеркально отражает дерево (меняет местами левых и правых детей)."""
    if not root:
        return None
        
    # Меняем местами детей текущего узла
    root.left, root.right = root.right, root.left
    
    # Рекурсивно продолжаем для поддеревьев
    invert_tree(root.left)
    invert_tree(root.right)
    
    return root

# Тест
print("Зеркальное отражение дерева:")
original_tree = build_tree([4, 2, 7, 1, 3, 6, 9])
print("Исходное дерево:")
print_tree(original_tree)

inverted = invert_tree(original_tree)
print("Зеркальное дерево:")
print_tree(inverted) 
# Ожидается: [4, 7, 2, 9, 6, 3, 1]

Зеркальное отражение дерева:
Исходное дерево:
['4', '2', '7', '1', '3', '6', '9']
Зеркальное дерево:
['4', '7', '2', '9', '6', '3', '1']
